# 第07章 C1组人工复核

G15／C1，首轮。提交快照 `019f7430fb3796f1abf4a8355de932d1e1b63b88`，历史 A1 习题版本与 A2 第二版另见[批判报告](C1组-批判报告.md)。

复现此文件只需 Python 3 和 NumPy。各复跑块明确注明抽取范围与种子，没有运行外部脚本、绘图或 MLP；不据此声称 A/B 的全章原 Notebook 已可复现。原作者写死的结论在 A2 原单元复跑中保留用于审查，不是 C1 结论。

本组借助 Codex 整理证据并运行验证代码。我们已完成人工复核，核对了关键推导和实验输出，并将结果汇总到本组的 C1 批判报告中。[独立推导](C1组-手动推导.md) · [AI 陷阱](C1组-AI陷阱.md)

In [1]:
import sys, platform, json
from math import comb, exp, sqrt, log, ceil, erf
from fractions import Fraction
import numpy as np

evidence = {}
print(f"Python {platform.python_version()} | NumPy {np.__version__} | {platform.system()} {platform.machine()} | CPU")

Python 3.12.14 | NumPy 2.3.5 | Windows AMD64 | CPU


## 7.1 有限类泛化界和 VC 代数检查

在有限固定类、独立同分布与有界损失下，样本量乘四时理论半径减半。下面用精确二项尾概率核验一个固定候选的偏差量级；有限类一致事件由并集界推出，不能由这次试验“证明”。另核对 A1 7.1.7 证明的半径方向（B1 T03）。

In [2]:
K, delta, p = 21, 0.05, 0.3
radii = [sqrt(log(2*K/delta)/(2*n)) for n in (20,80,320)]
assert np.allclose(radii[0], [radii[1]*2,radii[2]*4], rtol=0, atol=1e-14)
n=20
t=radii[0]
exact_tail=sum(comb(n,j)*p**j*(1-p)**(n-j) for j in range(n+1) if abs(j/n-p)>t)
single_bound=2*exp(-2*n*t*t)
assert exact_tail <= single_bound + 1e-14
print(f"有限类半径: {[round(x,8) for x in radii]}; 固定模型精确尾概率={exact_tail:.8f}; 上界={single_bound:.8f}")

n,d,delta=100,1,.05
A=d*log(2*exp(1)*n/d)+log(4/delta)
radius=sqrt(8*A/n)
at=exp(A+log(delta)-n*radius**2/8)
half=exp(A+log(delta)-n*(radius/2)**2/8)
assert abs(at-delta)<1e-12 and half>delta
print(f"VC尾界代数: 半径={radius:.9f}, 代入原式={at:.5f}, 半径减半时未截断上界={half:.4f}")
evidence['finite_vc']=[radii,exact_tail,single_bound,radius,half]

有限类半径: [0.41028654, 0.20514327, 0.10257164]; 固定模型精确尾概率=0.00004294; 上界=0.00238095
VC尾界代数: 半径=0.924352487, 代入原式=0.05000, 半径减半时未截断上界=150.5847


### 7.1 A1 源码定点复跑

试验从 A1 Markdown 中提取，不运行绘图。使用 11 个候选阈值、大测试集 20 万点和每档 200 次试验。原输出名 `true_risk` 实际是固定测试集估计，下面同解析风险分开：真实阈值为 0.5，独立翻转率为 0.1，故 $R(t)=0.1+0.8|t-0.5|$。

In [3]:
rng = np.random.default_rng(2026)
# 7.1 有限假设类的泛化界实验

def sample_threshold_data(n, rng, noise=0.1):
    x = rng.uniform(0, 1, n)
    y_clean = (x >= 0.5).astype(int)
    flip = rng.random(n) < noise
    y = np.where(flip, 1 - y_clean, y_clean)
    return x, y

thresholds = np.linspace(0, 1, 11)
H = [lambda x, t=t: (x >= t).astype(int) for t in thresholds]

# 固定大测试集
x_test, y_test = sample_threshold_data(200_000, rng)

def empirical_risk(h, x, y):
    return np.mean(h(x) != y)

def true_risk(h):
    return np.mean(h(x_test) != y_test)

n_values = [10, 20, 50, 100, 200, 500, 1000]
T = 200
delta = 0.05
log_term = np.log(len(H) / delta)

records = []
for n in n_values:
    emp_list, true_list, bound_list = [], [], []
    violations = 0
    for _ in range(T):
        x_tr, y_tr = sample_threshold_data(n, rng)
        risks = [empirical_risk(h, x_tr, y_tr) for h in H]
        best = int(np.argmin(risks))
        h_best = H[best]
        emp = risks[best]
        true = true_risk(h_best)
        bound = emp + np.sqrt(log_term / (2 * n))
        emp_list.append(emp)
        true_list.append(true)
        bound_list.append(bound)
        if true > bound:
            violations += 1
    records.append({
        "n": n,
        "emp_mean": np.mean(emp_list),
        "true_mean": np.mean(true_list),
        "true_std": np.std(true_list),
        "bound_mean": np.mean(bound_list),
        "violation_rate": violations / T,
    })

print(f"{'n':>6} {'emp_risk':>10} {'true_risk':>10} {'true_std':>9} {'bound':>9} {'viol_rate':>10}")
for r in records:
    print(f"{r['n']:>6} {r['emp_mean']:>10.4f} {r['true_mean']:>10.4f} "
          f"{r['true_std']:>9.4f} {r['bound_mean']:>9.4f} {r['violation_rate']:>10.3f}")

# 作图
ns = [r["n"] for r in records]
analytic = .1 + .8*np.abs(thresholds-.5)
fixed_test = np.array([true_risk(h) for h in H])
assert abs(analytic.min()-.1)<1e-14 and len(records)==7
print(f"候选0.5的解析风险/测试估计: {analytic[5]:.6f} / {fixed_test[5]:.6f}")
print(f"全候选测试估计与解析风险最大绝对差: {np.max(np.abs(fixed_test-analytic)):.6f}")
evidence['a1_71_last']=records[-1]
evidence['a1_71_test_vs_exact']=[float(analytic[5]),float(fixed_test[5])]

     n   emp_risk  true_risk  true_std     bound  viol_rate
    10     0.0875     0.1781    0.1003    0.6068      0.000
    20     0.0903     0.1309    0.0527    0.4575      0.000
    50     0.1045     0.1066    0.0224    0.3367      0.000
   100     0.1028     0.1010    0.0056    0.2670      0.000
   200     0.0992     0.1006    0.0000    0.2153      0.000
   500     0.0997     0.1006    0.0000    0.1732      0.000
  1000     0.0997     0.1006    0.0000    0.1517      0.000
候选0.5的解析风险/测试估计: 0.100000 / 0.100610
全候选测试估计与解析风险最大绝对差: 0.001870


## 7.2 ID/OOD 与 A1 外推实验

独立性不决定分布关系。同一训练域有多个可能延拓，单看训练域不能保证域外误差。A1 数值段用种子 2026 独立初始化，原代码在带噪数据上拟合、对无噪声 $\sin x$ 评价，省略绘图；此结果不冒充四节串行共享随机状态的运行。

In [4]:
f0=lambda x: 0
f1=lambda x: 8*max(0,x-1)**2
assert all(f0(i/10)==f1(i/10) for i in range(-10,11))
pstar=(f0(2)+f1(2))/2
assert max(abs(pstar-f0(2)),abs(pstar-f1(2)))==4
print('两延拓在 x=2 的真值:', f0(2), f1(2), '最坏绝对误差的最小值:', 4)
rng = np.random.default_rng(2026)
# 7.2 多项式回归的内插与外推

from numpy.polynomial import Polynomial

f = np.sin
sigma = 0.1
train_lo, train_hi = 0.0, 4.0
m = 50
N_eval = 200
T = 100

eval_intervals = {
    "内插区 [0.5,3.5]": (0.5, 3.5),
    "邻近外推 [4.5,6.0]": (4.5, 6.0),
    "远距外推 [7.0,9.0]": (7.0, 9.0),
}
degrees = [3, 5]

def rmse(y_pred, y_true):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

results = {deg: {k: [] for k in eval_intervals} for deg in degrees}
baseline_results = {k: [] for k in eval_intervals}

for _ in range(T):
    x_train = rng.uniform(train_lo, train_hi, m)
    y_train = f(x_train) + rng.normal(0, sigma, m)
    y_bar = np.mean(y_train)
    for name, (lo, hi) in eval_intervals.items():
        x_eval = np.linspace(lo, hi, N_eval)
        y_true = f(x_eval)
        baseline_results[name].append(rmse(np.full_like(y_true, y_bar), y_true))
        for deg in degrees:
            model = Polynomial.fit(x_train, y_train, deg=deg).convert()
            results[deg][name].append(rmse(model(x_eval), y_true))

print(f"{'区间':<22} {'三次均值':>10} {'三次标准差':>10} {'五次均值':>10} {'五次标准差':>10} {'基线均值':>10}")
for name in eval_intervals:
    r3 = np.array(results[3][name])
    r5 = np.array(results[5][name])
    rb = np.array(baseline_results[name])
    print(f"{name:<22} {r3.mean():>10.4f} {r3.std():>10.4f} {r5.mean():>10.4f} {r5.std():>10.4f} {rb.mean():>10.4f}")

# 作图
evidence['a1_72']={z:[float(np.mean(results[d][z])) for d in (3,5)] for z in results[3]}
assert all(len(results[3][z])==T for z in results[3])

两延拓在 x=2 的真值: 0 8 最坏绝对误差的最小值: 4


区间                           三次均值      三次标准差       五次均值      五次标准差       基线均值
内插区 [0.5,3.5]              0.0425     0.0099     0.0301     0.0121     0.4401
邻近外推 [4.5,6.0]             1.4008     0.4874     3.9427     2.6555     1.2087
远距外推 [7.0,9.0]             3.3647     2.2136    91.8583    59.4237     0.4523


## 7.3（补充）A1 的低秩解析参考

按 A1 源码取同一个 $N=200$ 三点差分矩阵。右端是参数的一次、二次项叠加，所以整个离散解族位于二维子空间内。预期用训练快照的前两个独立模式即可重建同一离散模型的域外解，离散解仍与连续真解有偏差。此处不训练 MLP，也不将 7.3 计作必做缺项。

In [5]:
N=200
x=np.linspace(0,1,N+2)[1:-1]; h=1/(N+1)
A=(2*np.eye(N)-np.eye(N,k=1)-np.eye(N,k=-1))/h**2
s=np.sin(np.pi*x); c=np.full(N,2.)
B=np.linalg.solve(A,np.column_stack([s,c]))
snap=B@np.array([[mu**2,mu] for mu in [.1,.2,.3,.4,.5]]).T
U,S,_=np.linalg.svd(snap,full_matrices=False)
rank=int(np.count_nonzero(S>S[0]*1e-12)); assert rank==2
Q=U[:,:rank]; mu=1.
b=mu**2*s+mu*c
fom=np.linalg.solve(A,b)
rom=Q@np.linalg.solve(Q.T@A@Q,Q.T@b)
exact=mu**2/np.pi**2*s+mu*x*(1-x)
rel=lambda u,v:float(np.linalg.norm(u-v)/np.linalg.norm(v))
r1,r2=rel(rom,fom),rel(rom,exact)
print(f"快照数值秩={rank}; μ=1 ROM/FOM相对误差={r1:.3e}; ROM/连续解误差={r2:.3e}")
assert r1<1e-10 and r2>r1
evidence['a1_73_rank']=[rank,r1,r2]

快照数值秩=2; μ=1 ROM/FOM相对误差=3.873e-14; ROM/连续解误差=5.738e-06


## 7.4 A1、A2 域外覆盖对照

A1 抽取数值段并以种子 2026 独立运行；每个配置重复 100 次训练/校准，复用固定 ID/OOD 测试集，并非 100 份独立测试集。A2 原 Notebook 零基索引 39，独立执行完整单元；原最后一行是写死的文本。B1 T06、B2 T08 的报告数据供比对。

In [6]:
rng = np.random.default_rng(2026)
# 7.4 分裂共形预测的覆盖率与宽度

from numpy.polynomial import Polynomial

f = np.sin
sigma = 0.1
n_train = 100
n_cal_values = [50, 200, 1000]
alphas = [0.20, 0.10, 0.05]
n_test = 2000
T = 100

def gen(n, lo, hi, rng):
    x = rng.uniform(lo, hi, n)
    y = f(x) + rng.normal(0, sigma, n)
    return x, y

# 固定测试集，保证跨重复可比
x_in, y_in = gen(n_test, 0, 2, rng)
x_out, y_out = gen(n_test, 3, 4, rng)

results = {}
for n_cal in n_cal_values:
    for alpha in alphas:
        cov_in, cov_out, wid_in, wid_out = [], [], [], []
        for _ in range(T):
            x_tr, y_tr = gen(n_train, 0, 2, rng)
            model = Polynomial.fit(x_tr, y_tr, deg=3).convert()
            x_cal, y_cal = gen(n_cal, 0, 2, rng)
            scores = np.abs(y_cal - model(x_cal))
            k = int(np.ceil((n_cal + 1) * (1 - alpha)))
            q = np.sort(scores)[k - 1] if k <= n_cal else np.inf
            for x_te, y_te, cov, wid in [
                (x_in, y_in, cov_in, wid_in),
                (x_out, y_out, cov_out, wid_out),
            ]:
                pred = model(x_te)
                cov.append(np.mean(np.abs(y_te - pred) <= q))
                wid.append(2 * q)
        results[(n_cal, alpha)] = {
            "cov_in": (np.mean(cov_in), np.std(cov_in)),
            "cov_out": (np.mean(cov_out), np.std(cov_out)),
            "wid_in": (np.mean(wid_in), np.std(wid_in)),
            "wid_out": (np.mean(wid_out), np.std(wid_out)),
        }

print(f"{'n_cal':>6} {'1-alpha':>8} {'cov_in':>8} {'cov_out':>8} {'width_in':>9} {'width_out':>10}")
for (n_cal, alpha), r in results.items():
    print(f"{n_cal:>6} {1-alpha:>8.2f} {r['cov_in'][0]:>8.3f} {r['cov_out'][0]:>8.3f} "
          f"{r['wid_in'][0]:>9.3f} {r['wid_out'][0]:>10.3f}")

# 作图
a1_example = results[(200,.1)]
evidence['a1_74']=[float(a1_example[k][0]) for k in ('cov_in','cov_out','wid_in','wid_out')]

 n_cal  1-alpha   cov_in  cov_out  width_in  width_out
    50     0.80    0.800    0.061     0.265      0.265
    50     0.90    0.903    0.079     0.347      0.347
    50     0.95    0.961    0.112     0.441      0.441
   200     0.80    0.802    0.060     0.264      0.264
   200     0.90    0.902    0.060     0.339      0.339
   200     0.95    0.951    0.117     0.405      0.405
  1000     0.80    0.802    0.045     0.263      0.263
  1000     0.90    0.900    0.090     0.336      0.336
  1000     0.95    0.951    0.086     0.401      0.401


In [7]:
# -*- coding: utf-8 -*-
"""
Created on Sat Oct 31 09:10:35 2026

@author: lenovo
"""

import numpy as np

# 设定随机种子
rng = np.random.default_rng(2026)

# 生成带噪 sin 数据
def generate_data(x_range, n, noise_std=0.3):
    x = rng.uniform(x_range[0], x_range[1], n)
    y = np.sin(2 * np.pi * x) + rng.normal(0, noise_std, n)
    return x, y

# 简单拟合模型（此处用真实函数 + 偏差代替，为了计算简便）
def model_predict(x):
    # 假设模型略微不准，带点整体偏移
    return np.sin(2 * np.pi * x) + 0.1 

# 计算 Split-Conformal 的 q
def compute_q(x_cal, y_cal, alpha=0.2):
    preds = model_predict(x_cal)
    residuals = np.abs(y_cal - preds)
    residuals.sort()
    n = len(residuals)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    # 防止 k 越界
    k = min(k, n) 
    return residuals[k - 1]

# 计算 PICP 和 MPIW
def evaluate_coverage(x_test, y_test, q):
    preds = model_predict(x_test)
    lower = preds - q
    upper = preds + q
    # 覆盖率 PICP
    picp = np.mean((y_test >= lower) & (y_test <= upper))
    # 平均宽度 MPIW
    mpiw = np.mean(upper - lower)
    return picp, mpiw

# 实验运行
alpha = 0.2
n_cal = 1000
n_test = 1000

# 1. 校准阶段 (域内)
x_cal, y_cal = generate_data([-1, 1], n_cal)
q = compute_q(x_cal, y_cal, alpha)

# 2. 测试阶段 (域内 vs 域外)
x_test_in, y_test_in = generate_data([-1, 1], n_test)
x_test_out, y_test_out = generate_data([2, 3], n_test)

picp_in, mpiw_in = evaluate_coverage(x_test_in, y_test_in, q)
picp_out, mpiw_out = evaluate_coverage(x_test_out, y_test_out, q)

print(f"设定的阈值 q = {q:.4f}")
print(f"域内测试: PICP = {picp_in:.4f}, MPIW = {mpiw_in:.4f}")
print(f"域外测试: PICP = {picp_out:.4f}, MPIW = {mpiw_out:.4f}")
print("结论：MPIW 没变（因为 q 没变），但域外 PICP 大幅塌陷，分布偏移破坏了可交换性假设。")

assert np.allclose([picp_in,picp_out,mpiw_in], [.813,.801,.821345573934], rtol=0, atol=1e-10)
assert abs(mpiw_in-mpiw_out)<1e-12
print(f"C1核验: ID-OOD差={100*(picp_in-picp_out):.1f}个百分点；不支持域外覆盖降至0.5以下。")
evidence['a2_74']=[float(q),float(picp_in),float(picp_out),float(mpiw_in),float(mpiw_out)]

设定的阈值 q = 0.4107
域内测试: PICP = 0.8130, MPIW = 0.8213
域外测试: PICP = 0.8010, MPIW = 0.8213
结论：MPIW 没变（因为 q 没变），但域外 PICP 大幅塌陷，分布偏移破坏了可交换性假设。
C1核验: ID-OOD差=1.2个百分点；不支持域外覆盖降至0.5以下。


### 分位边界、固定阈值覆盖与配对比较

无并列且可交换时，新分数秩均匀。用精确分数枚举校准数 4、5 的裁剪错误；再用 A2 的解析噪声分布核对两个域共享的覆盖概率。最后比较两个具有相同覆盖率差但不同逐点重合程度的测试情形，以核对 A1 历史解答题4。

In [8]:
for n in (4,5):
    k=ceil((n+1)*.9)
    ranks=range(1,n+2)
    clipped=Fraction(sum(r<=min(k,n) for r in ranks),n+1)
    conformal=Fraction(sum(r<=k for r in ranks),n+1)
    assert k==n+1 and clipped<Fraction(9,10) and conformal==1
    observed=compute_q(np.zeros(n),model_predict(np.zeros(n))+np.arange(1,n+1),.1)
    assert abs(observed-n)<1e-12
    print(f"n={n}; k={k}; A2实际返回={observed:g}；裁剪覆盖={clipped}，无穷阈值覆盖={conformal}")
cdf=lambda x:(1+erf(x/sqrt(2)))/2
theory=cdf((q+.1)/.3)-cdf((.1-q)/.3)
assert abs(theory-.80544505)<1e-7
print(f"A2 固定本次 q 的解析覆盖（ID 和 OOD 相同）: {theory:.8f}")
for plus,minus in [(4,0),(198,194)]:
    diff=np.array([1]*plus+[-1]*minus+[0]*(2000-plus-minus))
    print(f"配对不一致例: 覆盖差={diff.mean():.3f}; 差的标准误={diff.std(ddof=1)/sqrt(len(diff)):.6f}")
evidence['a2_analytic']=theory
print('C1_EVIDENCE_JSON='+json.dumps(evidence,ensure_ascii=False))

n=4; k=5; A2实际返回=4；裁剪覆盖=4/5，无穷阈值覆盖=1
n=5; k=6; A2实际返回=5；裁剪覆盖=5/6，无穷阈值覆盖=1
A2 固定本次 q 的解析覆盖（ID 和 OOD 相同）: 0.80544505
配对不一致例: 覆盖差=0.002; 差的标准误=0.000999
配对不一致例: 覆盖差=0.002; 差的标准误=0.009902
C1_EVIDENCE_JSON={"finite_vc": [[0.4102865429135277, 0.20514327145676384, 0.10257163572838192], 4.294002195359168e-05, 0.002380952380952379, 0.9243524869322057, 150.58470162377517], "a1_71_last": {"n": 1000, "emp_mean": 0.09974, "true_mean": 0.10061000000000002, "true_std": 1.3877787807814457e-17, "bound_mean": 0.15167085569462707, "violation_rate": 0.0}, "a1_71_test_vs_exact": [0.1, 0.10061], "a1_72": {"内插区 [0.5,3.5]": [0.042539579964141705, 0.03010538984579698], "邻近外推 [4.5,6.0]": [1.4007571109304784, 3.9427148945198507], "远距外推 [7.0,9.0]": [3.3646571483558603, 91.85826120424753]}, "a1_73_rank": [2, 3.873125376206518e-14, 5.7381558735649226e-06], "a1_74": [0.901735, 0.060285000000000005, 0.33935041234031443, 0.33935041234031443], "a2_74": [0.41067278696708664, 0.813, 0.801, 0.8213455739341734, 0.8213455739

## 复核结论和局限

- A1 实验代码已存在，但原文件是 Markdown 容器，必须改成可执行单元后再做全章复测。7.1 的测试估计需改名，7.2 的误差基准须明确，7.4 的概率保证不是每批实测比例的保证。
- A2 的域外实验没有出现正文声称的塌陷。A1 采用学习到的多项式预测器，A2 采用已知正弦函数加常数，结论不同是实验条件不同。A2 小样本分位裁剪另需修复。
- 我们的低秩测试只比较同一数学模型和离散模型下的结果，不能验证模型是否符合真实物理系统。本轮我们完成了人工复核，未重跑 A/B 组的整章实验。